In [1]:
import pandas as pd
import requests
import numpy as np
import time
import folium
import matplotlib.pyplot as plt
import json

In [2]:
# Load existing data if available
try:
    distance_df = pd.read_csv("data/osrm_distance_matrix_new.csv", index_col=0)
    distance_matrix = distance_df.values
    print(distance_df.shape)
    duration_df = pd.read_csv("data/osrm_duration_matrix_new.csv", index_col=0)
    duration_matrix = duration_df.values
    existing_codes = duration_df.index.tolist()  # Get existing codes from the index
except FileNotFoundError:
    print("No previous data found. Starting fresh.")
    distance_matrix = np.zeros((0, 0))  # Initialize empty matrices
    duration_matrix = np.zeros((0, 0))
    existing_codes = []

(758, 758)


In [3]:
# Load new data from CSV
day = "14-07-2025"
new_data = pd.read_csv(f"../data/test/orders-vol/{day}-PO-3.csv")
new_data = new_data.dropna(subset=['CODE'])


if np.issubdtype(new_data['CODE'].dtype, np.number):
    new_data['CODE'] = new_data['CODE'].astype(int)
    new_data['CODE'] = new_data['CODE'].astype(str)
    print('Converting to Object')

In [4]:
new_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CODE       37 non-null     object 
 1   SALE       37 non-null     float64
 2   VOLUME     37 non-null     float64
 3   DATE       37 non-null     object 
 4   LOCATION   37 non-null     object 
 5   ADDRESS    37 non-null     object 
 6   LATITUDE   37 non-null     float64
 7   LONGITUDE  37 non-null     float64
 8   BRAND      37 non-null     object 
 9   DISTRICT   37 non-null     object 
dtypes: float64(4), object(6)
memory usage: 3.0+ KB


In [5]:
(new_data['CODE']).dtype

dtype('O')

In [6]:
# # #Load new data from CSV
master_data = pd.read_csv('data/master_gps.csv')
master_data = master_data.dropna(subset=['CODE'])


In [7]:
SMAK_KADAWATHA = (7.0038321,79.9394804)

smak_data = {
    "CODE":'0',
    "LOCATION":"SMAK",
    "ADDRESS":"Smak, Kadawatha, Western Province, Sri Lanka",
    "LATITUDE":SMAK_KADAWATHA[0],
    "LONGITUDE":SMAK_KADAWATHA[1],
    "BRAND":"SMAK"
}


master_data = pd.concat(
    [
        pd.DataFrame(smak_data, index=[0]),
        master_data
    ],
    ignore_index=True
)

master_data = master_data.drop_duplicates(subset=['CODE'], keep='first')

In [8]:
print(new_data.columns)
print(master_data.columns)

Index(['CODE', 'SALE', 'VOLUME', 'DATE', 'LOCATION', 'ADDRESS', 'LATITUDE',
       'LONGITUDE', 'BRAND', 'DISTRICT'],
      dtype='object')
Index(['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND',
       'DISTRICT'],
      dtype='object')


In [9]:
master_data

,CODE,LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND,DISTRICT
0,0,SMAK,"Smak, Kadawatha, Western Province, Sri Lanka",7.003832,79.939480,SMAK,NaN
2,3,Wattala SC,"Wattala, Sri Lanka",6.990668,79.893171,Arpico,Gampaha
3,4,Borelasgamuwa SS,"Borelasgamuwa, Sri Lanka",6.840989,79.901719,Arpico,Colombo
4,5,Hyde Park SC,"Hyde, Sri Lanka",6.917587,79.858519,Arpico,Colombo
5,10,Nawinna SC,"Nawinna, Sri Lanka",6.853331,79.915072,Arpico,Colombo
...,...,...,...,...,...,...,...
717,1855,EX MIRISWATTA,MIRISWATTA,7.072739,80.015763,Cargills,Gampaha
718,1856,EX KIMBULAPITIYA,KIMBULAPITIYA,7.204241,79.894081,Cargills,Gampaha
719,1857,EX WILIMBULA,WILIMBULA,7.873054,80.771797,Cargills,Unknown
720,1861,EX KATANA,KATANA,7.248028,79.899366,Cargills,Gampaha


In [10]:
new_data = new_data[['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND','DISTRICT']]

In [11]:
updated_master = pd.concat(
    [master_data,
    new_data],
    ignore_index = True
).drop_duplicates(subset=['CODE'], keep='first')

In [ ]:
# updated_master.to_csv('data/master_gps.csv', index=False)

In [13]:
updated_master

,CODE,LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND,DISTRICT
0,0,SMAK,"Smak, Kadawatha, Western Province, Sri Lanka",7.003832,79.939480,SMAK,NaN
1,3,Wattala SC,"Wattala, Sri Lanka",6.990668,79.893171,Arpico,Gampaha
2,4,Borelasgamuwa SS,"Borelasgamuwa, Sri Lanka",6.840989,79.901719,Arpico,Colombo
3,5,Hyde Park SC,"Hyde, Sri Lanka",6.917587,79.858519,Arpico,Colombo
4,10,Nawinna SC,"Nawinna, Sri Lanka",6.853331,79.915072,Arpico,Colombo
...,...,...,...,...,...,...,...
753,Rathmalana,Rathmalana,Rathmalana,6.819545,79.880083,Laugf,Colombo
754,Seeduwa,Seeduwa,Seeduwa,7.124708,79.875003,Laugf,Gampaha
755,Wattala,Wattala,Wattala,6.990668,79.893171,Laugf,Gampaha
756,Wellawatta,Wellawatta,Wellawatta,6.875531,79.860998,Laugf,Colombo


In [14]:
# Extract locations and codes from new data
updated_master_codes = updated_master[['CODE', 'LATITUDE', 'LONGITUDE']].drop_duplicates(subset=['CODE'], keep='first')
updated_master_codes_list = updated_master_codes['CODE'].tolist()
updated_master_gps_list = [(row['LATITUDE'], row['LONGITUDE']) for _, row in updated_master_codes.iterrows()]

In [15]:
# Identify codes that are not in the existing matrix
codes_to_add = [code for code in updated_master_codes_list if code not in existing_codes]
locations_to_add = [loc for i, loc in enumerate(updated_master_gps_list) if updated_master_codes_list[i] in codes_to_add]

In [16]:
len(codes_to_add), len(locations_to_add)

(0, 0)

In [17]:
existing_locations = list(zip(updated_master['LATITUDE'], updated_master['LONGITUDE']))

In [18]:
len(existing_locations)

758

In [19]:
existing_locations

[(7.0038321, 79.9394804),
 (6.9906677, 79.8931709),
 (6.8409891, 79.9017187),
 (6.9175873, 79.8585189),
 (6.8533309, 79.9150724),
 (5.9496309, 80.5468529),
 (6.7106361, 79.9074262),
 (6.8411652, 79.9654324),
 (6.0328948, 80.2167912),
 (6.9060787, 79.9696277),
 (7.4817695, 80.3608876),
 (6.8624842, 79.8854983),
 (6.8432762, 80.0031833),
 (7.0864153, 80.0335107),
 (7.2513317, 80.3463754),
 (6.8758648, 79.9391936),
 (6.9906677, 79.8931709),
 (7.2657063, 79.8591235),
 (7.3096237, 80.7103532),
 (7.3359903, 80.6214005),
 (7.1842076, 79.9500477),
 (6.549971, 79.9836108),
 (7.1247085, 79.8750028),
 (6.9778284, 79.9271523),
 (7.0280037, 79.923),
 (6.8723185, 80.0003875),
 (7.0318781, 80.0283415),
 (7.472123, 80.0446221),
 (6.2441521, 80.0590804),
 (7.3493037, 79.8352767),
 (6.8261828, 79.881483),
 (7.0477897, 79.8970348),
 (6.92695, 79.9307072),
 (7.5619894, 79.8016569),
 (7.3529789, 80.6132646),
 (6.8472579, 80.0674619),
 (6.7055742, 80.3847345),
 (8.3113518, 80.4036508),
 (6.6235199, 80.54304

In [20]:
def get_osrm_data(origin, destination):
    """
    Get the distance between two coordinates using OSRM API.
    :param origin: (latitude, longitude)
    :param destination: (latitude, longitude)
    :return: Distance in meters
    """
    osrm_base_url = "http://router.project-osrm.org/route/v1/car"
    url = f"{osrm_base_url}/{origin[1]},{origin[0]};{destination[1]},{destination[0]}?overview=full&geometries=geojson"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        if "routes" in data and len(data["routes"]) > 0:
            path_cords = data["routes"][0]["geometry"]["coordinates"]
            distance = data["routes"][0]["distance"] / 1000
            duration = data["routes"][0]["duration"]/ 60
            return path_cords, distance, duration
        
    return None, np.inf, np.inf

In [21]:
if locations_to_add is not None:
    locations = existing_locations
    # locations.extend(locations_to_add)
    codes = existing_codes + codes_to_add
    num_codes = len(codes)
    print(len(locations))
    print(len(codes))
    # Resize matrices if new locations are added
    if distance_matrix.shape[0] < num_codes:
        old_size = distance_matrix.shape[0]
        new_distance_matrix = np.zeros((num_codes, num_codes))
        new_duration_matrix = np.zeros((num_codes, num_codes))
        
        if old_size > 0:
            new_distance_matrix[:old_size, :old_size] = distance_matrix
            new_duration_matrix[:old_size, :old_size] = duration_matrix
        
        distance_matrix = new_distance_matrix
        duration_matrix = new_duration_matrix
        

    # route_data = []
    for i in range(num_codes):
        for j in range(i + 1, num_codes):
            if distance_matrix[i][j] == 0 or np.isinf(distance_matrix[i][j]):
                origin = locations[i]
                destination = locations[j]

                path_cords, distance, duration = get_osrm_data(origin, destination)

                distance_matrix[i][j] = round(distance, 2)
                distance_matrix[j][i] = round(distance, 2)
                
                duration_matrix[i][j] = round(duration, 2)
                duration_matrix[j][i] = round(duration, 2)
                time.sleep(0.00001)
        print(f'{i} th row proceeded')

        if i % 5 == 0:
            distance_df = pd.DataFrame(distance_matrix, index=codes, columns=codes)
            distance_df.to_csv("data/osrm_distance_matrix_new.csv")
            duration_df = pd.DataFrame(duration_matrix, index=codes, columns=codes)
            duration_df.to_csv("data/osrm_duration_matrix_new.csv")
            
            # with open("data/osrm_route_data.json", "w") as json_file:
            #     json.dump(route_data, json_file, indent=4)

    # Final save for matrices and route data
    distance_df = pd.DataFrame(distance_matrix, index=codes, columns=codes)
    distance_df.to_csv("data/osrm_distance_matrix_new.csv")
    duration_df = pd.DataFrame(duration_matrix, index=codes, columns=codes)
    duration_df.to_csv("data/osrm_duration_matrix_new.csv")
    # with open("data/osrm_route_data.json", "w") as json_file:
    #     json.dump(route_data, json_file, indent=4)
    print(f"Added {len(locations_to_add)} new unique locations with codes: {codes_to_add}")
    print("Master GPS file updated.")       
else:
    print("No location to add")

758
758
0 th row proceeded
1 th row proceeded
2 th row proceeded
3 th row proceeded
4 th row proceeded
5 th row proceeded
6 th row proceeded
7 th row proceeded
8 th row proceeded
9 th row proceeded
10 th row proceeded
11 th row proceeded
12 th row proceeded
13 th row proceeded
14 th row proceeded
15 th row proceeded
16 th row proceeded
17 th row proceeded
18 th row proceeded
19 th row proceeded
20 th row proceeded
21 th row proceeded
22 th row proceeded
23 th row proceeded
24 th row proceeded
25 th row proceeded
26 th row proceeded
27 th row proceeded
28 th row proceeded
29 th row proceeded
30 th row proceeded
31 th row proceeded
32 th row proceeded
33 th row proceeded
34 th row proceeded
35 th row proceeded
36 th row proceeded
37 th row proceeded
38 th row proceeded
39 th row proceeded
40 th row proceeded
41 th row proceeded
42 th row proceeded
43 th row proceeded
44 th row proceeded
45 th row proceeded
46 th row proceeded
47 th row proceeded
48 th row proceeded
49 th row proceeded
50